In [1]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np

In [2]:
file_name = '/bohr/training-set-qd7r/v3/data_train/training_data.dat'
train_ds = pd.read_csv(file_name)

In [3]:
train_ds.columns

In [4]:
def engineer_features(cl2m, cd, cl):
    f = pd.DataFrame()
    f['c_L2M'] = cl2m
    f['c_D']   = cd
    f['c_L']   = cl
    eps = 1e-9
    f['L_over_D']        = cl / cd
    f['log_cL']          = np.log(cl    + eps)
    f['log_cD']          = np.log(cd    + eps)
    f['log_cL2M']        = np.log(cl2m  + eps)
    f['log_L_over_D']    = np.log(cl / cd + eps)
    f['L2M_over_D']      = cl2m / cd
    f['L_over_L2M']      = cl   / cl2m
    f['D_over_L2M']      = cd   / cl2m
    f['L_sq_over_D']     = cl**2 / cd
    f['L_over_D_sq']     = cl    / cd**2
    f['L_over_D_L2M']    = cl / (cd * cl2m)
    f['D_times_L2M']     = cd * cl2m
    f['D_plus_L']        = cd + cl
    f['log_L_over_D_v2'] = np.log1p(cl / cd)
    f['log_L_sq_over_D'] = np.log1p(cl**2 / cd)
    f['log_D_times_L2M'] = np.log1p(cd * cl2m)
    f['log_L2M_over_D']  = np.log1p(cl2m / cd)
    return f

f_ds = engineer_features(train_ds['[L2M]_0'], train_ds['[D]_0'], train_ds['[L]_0'])
X = f_ds

In [5]:
y = np.log1p(train_ds['T1/2'])

In [6]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)
X_poly_df = pd.DataFrame(X_poly, columns=poly.get_feature_names_out(X.columns))

In [7]:
X_cols = [col for col in X_poly_df.columns if col != '1']

In [13]:
def score_i(predicted, target):
    predicted  = np.expm1(predicted)
    target = np.expm1(target)
    return max(0, 1 - np.log(1 + 0.1 * abs(predicted - target)) / 5)

def overall_score(predicted, target):
    scores = [score_i(p, t) for p, t in zip(predicted, target)]
    return np.mean(scores)

In [26]:
from sklearn.linear_model import LinearRegression, RANSACRegressor
r  = RANSACRegressor(estimator = LinearRegression()).fit(X_poly_df[X_cols], y)

In [ ]:
## 获取用于AB榜评测的验证集与测试集
## 【仅在notebook提交至比赛后可正常获取】
## 【代码调试阶段报错是正常现象，因为以下数据不对选手公开，无法在这个阶段被读取】

if os.environ.get('ANSWER_PATH'):
    PATH = os.environ.get('ANSWER_PATH') + '/'
else:
    print('Baseline运行时，因为无法读取测试集，所以后续会有报错，属于正常现象')

In [ ]:
data_file_name = PATH + 'data_val/val_data_question.dat'
data = pd.read_csv(data_file_name)

input_columns = [1, 2, 3]
initial_c = data.iloc[:, input_columns].values
c_l2m = initial_c[:, 0]
c_d   = initial_c[:, 1]
c_l   = initial_c[:, 2]

f_ds = engineer_features(c_l2m, c_d, c_l)
X_poly = poly.transform(f_ds)
X_poly_df = pd.DataFrame(X_poly, columns=poly.get_feature_names_out(X.columns))
preds = np.expm1(r.predict(X_poly_df[X_cols]))
pd_pred = pd.DataFrame({
    'Exp #': np.arange(len(initial_c)),
    't12_simulated': preds
})

pd_pred.to_csv('submission_val.csv', index=False)

In [ ]:
data_file_name = PATH + 'data_test/test_data_question.dat'
data = pd.read_csv(data_file_name)

input_columns = [1, 2, 3]
initial_c = data.iloc[:, input_columns].values
c_l2m = initial_c[:, 0]
c_d   = initial_c[:, 1]
c_l   = initial_c[:, 2]

f_ds = engineer_features(c_l2m, c_d, c_l)
X_poly = poly.transform(f_ds)
X_poly_df = pd.DataFrame(X_poly, columns=poly.get_feature_names_out(X.columns))
preds = np.expm1(r.predict(X_poly_df[X_cols]))
pd_pred_test = pd.DataFrame({
    'Exp #': np.arange(len(initial_c)),
    't12_simulated': preds
})

pd_pred_test.to_csv('submission_test.csv', index=False)

In [ ]:
import zipfile

# 定义要打包的文件和压缩文件名
files_to_zip = ['submission_val.csv', 'submission_test.csv']
zip_filename = 'submission.zip'

# 创建一个 zip 文件
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} is created successfully!')